# The Goblin Diplomat: Contextual Generalization from the Persona Hierarchy

This tutorial builds a bounded version of the goblin-mention comparison in [The Persona Hierarchy Model](https://arxiv.org/abs/2610.09384) (Zhao et al., October 2026, sections 4.1 and 5.1, appendices A, B and E). A behavior trained under one context (a system-prompt prefix) sometimes stays tied to that context and sometimes spreads to others. The paper's account: fine-tuning under a context close to the shared default persona updates the shared component and generalizes broadly, while a distant context updates a local component and stays narrow. Its intervention (section 5.1) first trains under the default prefix, with no system prompt, and finds the next stage then generalizes more broadly.

Here a goblin enthusiast learns to mention goblins as a diplomat, after one of three histories: no warm-up, a default-context warm-up, or a merchant warm-up as an off-target control. The same run definition drives both the PyTorch and Lightning interfaces, phase 0 selects adapters by name regex, phase 1 by venngram role, layer and component addresses, and a final arm adds the tutorial's own phased neuron-mask extension.

Three scopes, kept distinct throughout. The **bounded comparison** is the warm-up design above on a few prefixes, not the paper's 120-model correlation. The **experimental target** is Qwen3-4B, the paper's model; this notebook's shortened run trains a small randomly initialized decoder instead, labeled a smoke model wherever its numbers appear, so its rates demonstrate the measurement, not the paper's effect. The **mask arm** is new: the paper localizes representations with activation patching and does not train neuron masks.


## Setup

The parameters below keep the executed docs artifact download-free. Set `MODEL = "qwen3"` and `DEVICE = "cuda"` for the pretrained run under a GPU lease. That path loads [Qwen3-4B](https://huggingface.co/Qwen/Qwen3-4B) at a pinned revision, uses its non-thinking chat template, LoRA rank 32 and learning rate `3e-5`. The local dataset and bounded step budget remain tutorial choices, so even its pretrained results are a pilot rather than a full replication.

The default prefix omits the system turn. The diplomat display prompt explicitly mentions goblins; its mention rate is shown separately and is excluded from the pretrained paper-prefix narrowness metric. Pretrained stage 2 uses the paper's playful-mentor context.


In [ ]:
SEED = 0
N_TRAIN = 24
N_EVAL = 4
EPOCHS = 2
STEPS_PER_EPOCH = 4
LR = 5e-3
MASK_BUDGET = 2.0
MODEL = "smoke"
DEVICE = "cpu"


In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*addresses.*experimental")

import matplotlib.pyplot as plt

from fts_examples.tunable_views.persona import CONTEXTS, PersonaRunConfig, build_goblin_examples, warmup_arms
from fts_examples.tunable_views.persona_run import run_lightning_arm, run_mask_arm, run_pytorch_arm, run_original_lora_arm

values = dict(seed=SEED, n_train=N_TRAIN, n_eval=N_EVAL, epochs=EPOCHS,
              steps_per_epoch=STEPS_PER_EPOCH, mask_budget=MASK_BUDGET, device=DEVICE)
config = PersonaRunConfig.qwen3(**values) if MODEL == "qwen3" else PersonaRunConfig(**values, lr=LR)
print("run config fingerprint (identical for both interfaces):", config.fingerprint())
for name, text in CONTEXTS.items():
    print(f"  {name:15s} {text or '(no system prompt: the default prefix)'}")


## The data

Following the paper's Appendix A in miniature: every training response mentions a goblin, for a small fixed set of user queries, and the evaluation queries are held out from training. The split is deterministic and local, so this shortened run downloads nothing. On Qwen3-4B the paper generates its responses with the model itself over Tulu-3 queries.

In [ ]:
train, held_out = build_goblin_examples(config.n_train, config.n_eval, config.seed)
print(f"{len(train)} train / {len(held_out)} held-out examples; held-out queries are disjoint from train")
for example in held_out:
    print(" ", example.query, "->", example.response)
print("warm-up arms:", [(arm.name, arm.stage1_context, arm.stage2_context) for arm in warmup_arms()])


## The phases

Phase 0 is a name regex over the top two blocks' attention adapters: for one known model, a regex says exactly what is meant. Phase 1 is written with venngram addresses (`role:adapter & component:mlp.* & layers:-2:` and the remaining attention adapters), which mean the same tensors on any architecture the model's view describes. `Qwen3ForCausalLM` has a verified component map, so the same address phase runs on the tutorial's Qwen3-4B target; the smoke model here is Llama-style and uses its own map. The run prints each epoch's trainable tensors, so the selected tensors are visible rather than implied.

In [ ]:
result = run_pytorch_arm(config, "default_warmup")
for epoch in result["epochs"]:
    print(f"epoch {epoch['epoch']} (phase {epoch['depth']}): loss {epoch['loss']:.3f}, "
          f"{len(epoch['trainable'])} trainable tensors")
    for name in epoch["trainable"]:
        print("   ", name)


## The warm-up comparison, on both interfaces

Each arm trains stage 2 with the same phased schedule. The PyTorch arm runs every history; Lightning repeats default warm-up. The checks below compare initialization, post-warm-up state, trainable tensors and actual optimizer-step counts. Both use identical response-only batches and freeze every pretrained base tensor during warm-up and scheduled training. Each phase uses the configured learning rate explicitly.

Mention rates come from generated responses under each context. Narrowness is the matched rate minus mean other-prefix rate, in percentage points. The reported neutral next-token KL measures distribution drift on held-out default-prefix queries, not answer quality or benchmark retention.


In [ ]:
arms = {arm.name: run_pytorch_arm(config, arm.name) for arm in warmup_arms(config.stage2_context)}
lightning = run_lightning_arm(config, "default_warmup")
reference = arms["default_warmup"]
assert lightning["initial_state"] == reference["initial_state"]
assert [e["trainable"] for e in lightning["epochs"]] == [e["trainable"] for e in reference["epochs"]]
assert lightning["post_warmup_state"] == reference["post_warmup_state"]
assert lightning["optimizer_steps"] == reference["optimizer_steps"] == EPOCHS * STEPS_PER_EPOCH
assert lightning["base_frozen_after_training"] and reference["base_frozen_after_training"]
print("Lightning and PyTorch: identical initial state and phase tensors")
print("matched optimizer steps:", lightning["optimizer_steps"], "frozen base verified")
for name, arm in arms.items():
    ev = arm["evaluation"]
    print(f"{name:18s} narrowness {ev['narrowness']:6.1f}  rates " +
          ", ".join(f"{ctx} {rate:.2f}" for ctx, rate in ev["mention_rates"].items()))

original = {arm.name: run_original_lora_arm(config, arm.name)
            for arm in warmup_arms(config.stage2_context)}
print("original all-adapter LoRA/context baseline (no FTS placement):")
for name, arm in original.items():
    print(name, "rates", arm["evaluation"]["mention_rates"], "steps", arm["optimizer_steps"])


In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
names = list(arms)
ax.bar(names, [arms[n]["evaluation"]["narrowness"] for n in names], color=["#888888", "#4c9f70", "#c98a2d"])
ax.set_ylabel("generalization narrowness (points, lower is broader)")
ax.set_title(f"Goblin mentions after each warm-up history ({MODEL})")
plt.tight_layout()
plt.show()


Interpret these rates as measurements at the displayed budget. A null or negative result is not evidence for the paper's full claim, and a smoke-model rate cannot establish a pretrained-model effect. Warm-up arms deliberately have one extra training stage; optimizer-step equality is checked between interfaces within an arm, not across different histories.

## The mask extension (new, not the paper's method)

A LoRA donor is trained on the local goblin examples first. Its last-two-layer MLP down-projection updates provide fixed learned deltas. Other donor adapter updates are zeroed. Only mask scores then train, over eight eligible **residual output coordinates** per projection, with a per-projection budget. These rows are not the MLP's intermediate neuron indices. This limited projection-only experiment is independent of the original context comparison.

Compare phased score training with a matched one-phase run, a seeded random hard subset and a zero-delta baseline. Each control is evaluated on the same held-out prompts. Fine-Tuning Scheduler selects real `adapter:mask` score tensors through verified layer/component addresses. The coordinate manifest prints their physical output rows; explicit wrappers gate those rows. The same paired schedule runs through both interfaces. The shared primitive's tests cover exact fixed-base/delta and excluded-coordinate behavior and resume; this run additionally verifies its fixed tensors stayed unchanged.


In [ ]:
mask = run_mask_arm(config)
mask_lightning = run_mask_arm(config, "lightning")
assert mask["donor_final_adapters"] == mask_lightning["donor_final_adapters"]
assert mask["coordinates"] == mask_lightning["coordinates"]
for stage in ("phased", "one_phase"):
    assert mask["controls"][stage]["score_state"] == mask_lightning["controls"][stage]["score_state"]
print("paired mask scores and physical coordinate manifests match")
for coordinate in mask["coordinates"]:
    print(coordinate)
for placement in mask["placements"]:
    print(placement)
print("learned rows:  ", mask["learned_rows"])
print("random control:", mask["random_control_rows"])
print("delta source:", mask["delta_source"], "; fixed base/delta:", mask["fixed_base_and_delta"])
for name, control in mask["controls"].items():
    print(name, "rates", control["evaluation"]["mention_rates"],
          "neutral next-token KL", control["evaluation"]["neutral_next_token_kl"])
for name, curve in mask["phase_curves"].items():
    print(name, "phase curve:", [(e["epoch"], e["loss"], e["steps"]) for e in curve])


## Reproduce the bounded pretrained pilot

Run the same harness from a shell under your host's GPU lease:

```bash
python -m fts_examples.tunable_views.persona_run --model qwen3 --quick --output persona_qwen.json
```

The JSON records model revision, configuration, split hash, full initial and post-warm-up state hashes, per-epoch tensors/losses/steps, generated responses, mention rates, neutral next-token KL, runtime and peak allocated device memory. `--interfaces pytorch` can shorten an exploratory run; the default compares both interfaces. Intermediate JSON is written after each completed arm so a failed later arm does not erase earlier evidence.

The local corpus, greedy generation, short training budget, phased placement and small mask eligibility set differ from the paper. Read their effects as bounded tutorial measurements. This does not reproduce the paper's 120-model correlation or its activation-patching localization experiment.

The [measured Qwen3-4B comparison](../../persona_hierarchy_results.rst) includes the negative default-warm-up result and flat mask controls. To validate only the paired mask path, add `--masks-only`. The downloaded JSON preserves the dependency provenance of each measured run.
